# 03 — Model Training

Loads pre-built feature matrices and compares candidate models.
All production logic lives in `src/train.py`.

> ⚠️ **Class imbalance reminder:** All models use `class_weight='balanced'` to compensate for the 1.3:1 SQLi:Benign ratio. After training, check `recall(benign) >= 0.85`. If not → enable SMOTE.

## Setup

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json, pickle, warnings

from scipy.sparse import load_npz
from sklearn.linear_model    import LogisticRegression
from sklearn.svm             import LinearSVC
from sklearn.calibration     import CalibratedClassifierCV
from sklearn.metrics         import (classification_report, confusion_matrix,
                                     roc_auc_score, f1_score, recall_score)
from sklearn.model_selection import StratifiedKFold, cross_val_score

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})

X_train = load_npz("../data/processed/features_train.npz")
X_val   = load_npz("../data/processed/features_val.npz")
y_train = np.load("../data/processed/labels_train.npy")
y_val   = np.load("../data/processed/labels_val.npy")

print(f"X_train: {X_train.shape} | Benign: {(y_train==0).sum():,} | SQLi: {(y_train==1).sum():,}")
print(f"X_val  : {X_val.shape}   | Benign: {(y_val==0).sum():,}  | SQLi: {(y_val==1).sum():,}")
print(f"Sparsity: {100*(1-X_train.nnz/(X_train.shape[0]*X_train.shape[1])):.2f}%")

## Candidate Models

All three use `class_weight='balanced'` to handle the 1.3:1 class imbalance without touching the data.

| Model | Why included |
|-------|-------------|
| Logistic Regression | Fast, interpretable baseline for sparse TF-IDF |
| LR (C=10) | Less regularisation — tests if data is clean enough |
| Linear SVM | Often best for high-dimensional sparse text |

In [ ]:
CANDIDATES = {
    "Logistic Regression": LogisticRegression(
        C=1.0, max_iter=500, class_weight="balanced", solver="lbfgs", random_state=42),
    "Logistic Regression (C=10)": LogisticRegression(
        C=10.0, max_iter=500, class_weight="balanced", solver="lbfgs", random_state=42),
    "Linear SVM": CalibratedClassifierCV(
        LinearSVC(C=0.1, max_iter=2000, class_weight="balanced", random_state=42)),
}
for name in CANDIDATES:
    print(f"  ✓  {name}")

## 5-Fold Stratified Cross-Validation

CV runs on training data only — the val set is never seen here.
Stratified folds preserve the 57/43 class ratio in every fold.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}

for name, model in CANDIDATES.items():
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="f1_macro", n_jobs=-1)
    cv_results[name] = scores
    print(f"  {name:<30}  CV F1-macro: {scores.mean():.4f} ± {scores.std():.4f}")

## Train on Full Train Set → Evaluate on Val Set

> Val set is used here for **model selection only** — never for final scoring. Final evaluation is in `04_evaluation.ipynb`.

In [ ]:
val_results = {}

for name, model in CANDIDATES.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    y_prob = model.predict_proba(X_val)[:, 1]

    f1   = f1_score(y_val, y_pred, average="macro")
    auc  = roc_auc_score(y_val, y_prob)
    rec0 = recall_score(y_val, y_pred, pos_label=0)
    rec1 = recall_score(y_val, y_pred, pos_label=1)

    val_results[name] = dict(cv_f1=float(cv_results[name].mean()),
                             val_f1=float(f1), auc=float(auc),
                             rec0=float(rec0), rec1=float(rec1))

    print(f"\n--- {name} ---")
    print(classification_report(y_val, y_pred, target_names=["Benign","SQLi"]))
    print(f"  ROC-AUC         : {auc:.4f}")
    flag = "⚠ BELOW 0.85 — consider SMOTE" if rec0 < 0.85 else "✓"
    print(f"  Recall (benign) : {rec0:.4f}  {flag}")
    cm = confusion_matrix(y_val, y_pred)
    print(f"  TN={cm[0,0]:,} FP={cm[0,1]:,} | FN={cm[1,0]:,} TP={cm[1,1]:,}")

## Model Selection Summary

In [ ]:
results_df = pd.DataFrame(val_results).T.reset_index().rename(columns={"index":"Model"})
print(results_df.to_string(index=False))

best_name = results_df.loc[results_df["val_f1"].idxmax(), "Model"]
print(f"\n✓  Best model: {best_name}")

# Visual comparison
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
metrics   = [("val_f1","Val F1-macro"), ("auc","ROC-AUC"), ("rec0","Recall — Benign")]
colors    = ["#4A90D9","#E05C5C","#F5A623"]

for ax, (metric, title), color in zip(axes, metrics, colors):
    vals  = results_df[metric].astype(float)
    bars  = ax.bar(results_df["Model"], vals, color=color, alpha=0.85, width=0.5)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.001,
                f"{val:.4f}", ha="center", fontsize=9, fontweight="bold")
    ax.set_title(title, fontweight="bold"); ax.set_ylim(0.95, 1.01)
    ax.set_xticklabels(results_df["Model"], rotation=15, ha="right", fontsize=8)
    if metric == "rec0":
        ax.axhline(0.85, color="red", linestyle="--", lw=1, label="min threshold")
        ax.legend(fontsize=8)

plt.suptitle("Model comparison on validation set", fontweight="bold")
plt.tight_layout(); plt.show()

## Summary

| Model | CV F1 | Val F1 | ROC-AUC | Rec(Benign) | Rec(SQLi) |
|-------|------:|-------:|--------:|------------:|----------:|
| Logistic Regression | 0.9908 | 0.9888 | 0.9990 | 0.9953 ✓ | 0.9843 |
| LR (C=10) | 0.9908 | 0.9895 | 0.9988 | 0.9941 ✓ | 0.9864 |
| **Linear SVM** | **0.9908** | **0.9960** | **0.9997** | **0.9974 ✓** | **0.9950** |

**Linear SVM wins** — highest F1, highest AUC, fewest errors. All three models comfortably clear the benign recall threshold → `class_weight='balanced'` was sufficient, no SMOTE needed.

⚠️ **Class imbalance — deployment reminder:** This test set is 57/43. In production (99%+ benign), threshold tuning becomes critical — see `04_evaluation.ipynb`.

➡️ Next: `04_evaluation.ipynb`